# Multi-Agent Systems

Creating a multi-agent is as simple as configuring a sub-agent as tool in a master orchestrator agent.

In [54]:
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import HumanMessage

from dotenv import load_dotenv

In [55]:
load_dotenv()

True

#### Setting up the normal tools for the agent

In [56]:
@tool
def square_root(x: float) -> float:
    """Finds the square root of a number"""
    return x ** 0.5

@tool
def square(x: float) -> float:
    """Finds the square of a given number"""
    return x ** 2

#### Defining the sub-agents

In [57]:
subagent1 = create_agent(
    model='claude-haiku-4-5',
    tools=[square_root]
)

subagent2 = create_agent(
    model='claude-haiku-4-5',
    tools=[square]
)

#### Defining the tools for the main agent to call the subagents

Each agent was defined with the relevant tools above, and tools for the main agent are created below to allow the invocation of the subagents to perform a specific task. The subagents are passed as tools when defining the main agent.

In [58]:
@tool
def call_subagent1(x: float) -> float:
    """Call subagent 1 to calculate the square root of a number"""
    response = subagent1.invoke({"messages": [HumanMessage(content=f"Calculate the square root of {x}")]})
    return response["messages"][-1].content

@tool
def call_subagent2(x: float) -> float:
    """Call subagent 1 to calculate the square of a number"""
    response = subagent2.invoke({"messages": [HumanMessage(content=f"Calculate the square of {x}")]})
    return response["messages"][-1].content

In [59]:
orchestrator_agent = create_agent(
    model='claude-sonnet-5',
    tools=[call_subagent1, call_subagent2],
    system_prompt="""You are a helpful mathematical assistant.
    Use the subagents to answer key questions."""
)

In [60]:
orchestrator_agent.invoke({"messages": [HumanMessage(content="What's the square of 25?")]})

{'messages': [HumanMessage(content="What's the square of 25?", additional_kwargs={}, response_metadata={}, id='ee4db33a-4fc9-44c7-9770-1afd1a2dd340'),
  AIMessage(content=[{'id': 'toolu_01WMsmZWEC6w9BWRAYTLHw4o', 'caller': {'type': 'direct'}, 'input': {'x': 25}, 'name': 'call_subagent2', 'type': 'tool_use'}], additional_kwargs={}, response_metadata={'id': 'msg_011CeBE2WK6jqqEjqwQ6FGKR', 'container': None, 'model': 'claude-sonnet-5', 'stop_details': None, 'stop_reason': 'tool_use', 'stop_sequence': None, 'usage': {'cache_creation': {'ephemeral_1h_input_tokens': 0, 'ephemeral_5m_input_tokens': 0}, 'cache_creation_input_tokens': 0, 'cache_read_input_tokens': 0, 'inference_geo': 'global', 'input_tokens': 561, 'output_tokens': 50, 'output_tokens_details': {'thinking_tokens': 0}, 'server_tool_use': None, 'service_tier': 'standard'}, 'model_name': 'claude-sonnet-5', 'model_provider': 'anthropic'}, id='lc_run--01a01798-772c-7d43-bfce-5e4600d4381a-0', tool_calls=[{'name': 'call_subagent2', 'arg